# Visium Data Preprocessing (MPP=0.5 기준)

Visium 기술만 대상으로  
MPP=0.5 µm/pixel (20x) 기준 grid 패치 추출 + **3-head label** 생성

### 3-Head Label Design
- **Head A: Presence** — gene이 패치에 존재하는지 (binary, BCE loss)
- **Head B: Expression** — 존재 시 조건부 발현량 (log1p normalized, masked Gaussian NLL)
- **Head C: Uncertainty** — 예측 불확실성 (학습 시 자동 학습, label 불필요)

In [ ]:
# ===== 1. 설정 및 라이브러리 =====
from glob import glob
import os
import numpy as np
import pandas as pd
import openslide
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import anndata
import warnings
warnings.filterwarnings('ignore', message='Variable names are not unique')

def create_dir(path):
    os.makedirs(path, exist_ok=True)

# 하이퍼파라미터
TARGET_MPP = 0.5        # 20x 해상도 (µm/pixel)
CONTEXT_SCALE = 4       # 5x context = 4x wider
PATCH_SIZE = 512        # 패치 크기 (pixels)
MIN_TRANSCRIPTS = 10    # 패치 내 최소 transcript 수
MIN_GENES = 10000       # 슬라이드 최소 gene 수 (이하 제외)

# 대상 기술
TARGET_TECHS = ['Visium','Visium HD', "Visium HD 3'"]

print(f'TARGET_MPP: {TARGET_MPP} µm/pixel')
print(f'PATCH_SIZE: {PATCH_SIZE}')
print(f'CONTEXT_SCALE: {CONTEXT_SCALE} (5x)')
print(f'MIN_TRANSCRIPTS: {MIN_TRANSCRIPTS}')
print(f'MIN_GENES: {MIN_GENES}')
print(f'TARGET_TECHS: {TARGET_TECHS}')

In [ ]:
# ===== 2. 데이터 경로 및 저장 디렉토리 생성 =====
data_path = '../../data/spatialTranscriptome/'
parquet_path = f'{data_path}transcripts/'  # parquet transcript 파일 경로
save_base = '../../data/visium_marker_gene_spatial_expression'

meta_df = pd.read_csv(f'{data_path}meta_df_homo_sapiens.csv')
print(f'Total slides in metadata: {len(meta_df)}')

# 대상 기술만 필터링
target_df = meta_df[meta_df['st_technology'].isin(TARGET_TECHS)].reset_index(drop=True)
print(f'Target slides ({TARGET_TECHS}): {len(target_df)}')
print(target_df['st_technology'].value_counts())

# parquet 파일 존재 여부 확인
parquet_available = []
for idx in range(len(target_df)):
    sid = target_df.iloc[idx]['id']
    pq_file = f'{parquet_path}{sid}_transcripts.parquet'
    parquet_available.append(os.path.exists(pq_file))
target_df['has_parquet'] = parquet_available
print(f'Slides with parquet: {sum(parquet_available)} / {len(target_df)}')

# organ 목록
organs = target_df['organ'].unique().tolist()
print(f'Organs: {organs}')

# 저장 디렉토리 생성
for o in organs:
    for t in TARGET_TECHS:
        create_dir(f'{save_base}/image/{t}/{o}')
        create_dir(f'{save_base}/image_5x/{t}/{o}')
        create_dir(f'{save_base}/label_presence/{t}/{o}')
        create_dir(f'{save_base}/label_expression/{t}/{o}')

print('Directories created.')

In [ ]:
# ===== 3. 전체 슬라이드 h5ad gene 목록 수집 =====
import re

h5ad_path = f'{data_path}st/'

# 의미없는 gene 필터링 패턴
EXCLUDE_PREFIXES = (
    'MT-',       # 미토콘드리아 유전자
    'RPL', 'RPS', # 리보솜 단백질
    'LINC',      # long intergenic non-coding RNA
    'LOC',       # uncharacterized locus
    'MIR',       # microRNA
    'SNOR', 'SNOU',  # small nucleolar RNA
    'RNA5S', 'RNA18S', 'RNA28S', 'RNA45S',  # ribosomal RNA
    'MTRNR',     # mitochondrial rRNA
)
EXCLUDE_PATTERNS = re.compile(
    r'^(AC\d{6}|AL\d{6}|AP\d{6}|BX\d{6}|CR\d{6})'  # Ensembl novel transcripts
    r'|(-AS\d+$|-IT\d+$|-DT$)'    # antisense / intronic / divergent transcript
    r'|(^ENSG\d+$)'               # Ensembl ID (이름 미해석)
)

def is_meaningful_gene(gene_name):
    """의미 있는 protein-coding gene인지 판별"""
    if gene_name.startswith(EXCLUDE_PREFIXES):
        return False
    if EXCLUDE_PATTERNS.search(gene_name):
        return False
    # 소문자가 포함된 gene name 제외 (orf, pseudogene 등)
    if any(c.islower() for c in gene_name):
        return False
    return True

slide_gene_sets = {}  # sample_id → set of gene names
skipped_low_genes = []

for idx in tqdm(range(len(target_df)), desc='Collecting gene names from h5ad'):
    row = target_df.iloc[idx]
    sample_id = row['id']

    h5ad_file = f'{h5ad_path}{sample_id}.h5ad'
    if not os.path.exists(h5ad_file):
        continue

    adata = anndata.read_h5ad(h5ad_file, backed='r')
    var_names = adata.var_names.tolist()
    adata.file.close()

    # 타입 확인 및 str 변환 (bytes → decode)
    types = set(type(g).__name__ for g in var_names)
    if types != {'str'}:
        print(f'  {sample_id}: gene name types = {types}, converting via decode')
        var_names = [g.decode('utf-8') if isinstance(g, bytes) else str(g) for g in var_names]

    # 의미없는 gene 필터링
    gene_names = {g for g in var_names if is_meaningful_gene(g)}

    if len(gene_names) <= MIN_GENES:
        skipped_low_genes.append((sample_id, len(gene_names)))
        continue

    slide_gene_sets[sample_id] = gene_names

# 필터링 효과 확인 (첫 슬라이드 기준)
if slide_gene_sets:
    first_sid = next(iter(slide_gene_sets))
    adata_tmp = anndata.read_h5ad(f'{h5ad_path}{first_sid}.h5ad', backed='r')
    raw_count = len(adata_tmp.var_names)
    adata_tmp.file.close()
    filtered_count = len(slide_gene_sets[first_sid])
    print(f'\n필터링 예시 ({first_sid}): {raw_count:,} → {filtered_count:,} genes ({raw_count - filtered_count:,}개 제거)')

print(f'\nh5ad 로드 성공: {len(slide_gene_sets)} / {len(target_df)} slides')
print(f'Gene 수 {MIN_GENES}개 이하로 제외: {len(skipped_low_genes)}개')
for sid, n in skipped_low_genes:
    print(f'  {sid}: {n:,} genes')
if slide_gene_sets:
    print(f'Gene 수 범위: {min(len(g) for g in slide_gene_sets.values()):,} ~ {max(len(g) for g in slide_gene_sets.values()):,}')

    all_types = set()
    for genes in slide_gene_sets.values():
        all_types.update(type(g).__name__ for g in genes)
    print(f'Gene name 타입: {all_types}')

In [ ]:
# ===== 4. 85% 이상 슬라이드에 존재하는 gene 추출 =====
from collections import Counter

COVERAGE_THRESHOLD = 0.85  # 85% 이상

# gene별 등장 슬라이드 수 카운트
gene_counter = Counter()
for genes in slide_gene_sets.values():
    gene_counter.update(genes)

n_slides = len(slide_gene_sets)
min_slides = int(np.ceil(n_slides * COVERAGE_THRESHOLD))

common_genes = {g for g, cnt in gene_counter.items() if cnt >= min_slides}
common_genes_sorted = sorted(common_genes)

print(f'전체 슬라이드 수: {n_slides}')
print(f'기준: {COVERAGE_THRESHOLD*100:.0f}% 이상 = {min_slides}개 슬라이드 이상 보유')
print(f'선택된 gene 수: {len(common_genes):,}')
print(f'\n선택된 gene 목록 (처음 50개):')
for i, g in enumerate(common_genes_sorted[:50]):
    cnt = gene_counter[g]
    print(f'  [{i:>4d}] {g:>15s}  ({cnt}/{n_slides} slides, {cnt/n_slides*100:.1f}%)')
if len(common_genes_sorted) > 50:
    print(f'  ... 외 {len(common_genes_sorted) - 50}개')

# 커버리지별 gene 수 분포
print(f'\n커버리지별 gene 수:')
for pct in [100, 95, 90, 85, 80, 70, 50]:
    threshold = int(np.ceil(n_slides * pct / 100))
    n_genes = sum(1 for cnt in gene_counter.values() if cnt >= threshold)
    print(f'  {pct:>3d}% ({threshold:>3d}+ slides): {n_genes:,} genes')

# 선택된 common_genes를 보유하지 않는 슬라이드 확인
print(f'\n===== 버려지는 슬라이드 (common_genes 중 누락 있는 슬라이드) =====')
dropped_slides = []
for sid, genes in slide_gene_sets.items():
    missing = common_genes - genes
    if missing:
        dropped_slides.append((sid, len(missing), sorted(missing)))

print(f'common_genes 완전 보유: {n_slides - len(dropped_slides)} / {n_slides} slides')
print(f'누락 있는 슬라이드: {len(dropped_slides)}개')
for sid, n_miss, miss_list in sorted(dropped_slides, key=lambda x: -x[1]):
    print(f'  {sid}: {n_miss}개 누락 — {miss_list[:10]}{"..." if n_miss > 10 else ""}')

In [ ]:
# ===== 5. 누락 슬라이드의 missing gene 제거 + CSV 저장 =====

# 누락 슬라이드들이 빠뜨린 gene 모두 수집
genes_to_remove = set()
for sid, n_miss, miss_list in dropped_slides:
    genes_to_remove.update(miss_list)

print(f'누락 슬라이드에서 빠진 gene 수: {len(genes_to_remove)}')
print(f'제거 전 common_genes: {len(common_genes):,}')

# common_genes에서 제거
common_genes -= genes_to_remove
common_genes_sorted = sorted(common_genes)

print(f'제거 후 common_genes: {len(common_genes):,}')

# gene CSV 저장
save_csv_path = f'{save_base}/visium_common_genes.csv'
df_genes = pd.DataFrame({
    'gene': common_genes_sorted,
    'coverage': [gene_counter[g] for g in common_genes_sorted],
    'coverage_pct': [gene_counter[g] / n_slides * 100 for g in common_genes_sorted],
})
df_genes.to_csv(save_csv_path, index=False)
print(f'\ngene CSV 저장 완료: {save_csv_path}')
print(f'총 {len(df_genes)} genes')
print(df_genes.head(10))

# 사용 슬라이드 목록 (missing gene 제거 후 모든 슬라이드가 common_genes 보유)
valid_slides = list(slide_gene_sets.keys())

df_slides = target_df[target_df['id'].isin(valid_slides)][['id', 'st_technology', 'organ', 'image_filename']].reset_index(drop=True)
save_slides_path = f'{save_base}/visium_valid_slides.csv'
df_slides.to_csv(save_slides_path, index=False)

print(f'\n사용 슬라이드 CSV 저장 완료: {save_slides_path}')
print(f'총 {len(df_slides)} / {n_slides} slides')
print(df_slides['st_technology'].value_counts().to_string())
print(df_slides['organ'].value_counts().to_string())